# Notebook 05 - Tuning des hyperparametres
Optimisation par RandomizedSearchCV du meilleur couple identifie en modelisation.

La grille couvre des profondeurs moderees pour limiter le sur-apprentissage,
plusieurs learning rates, 300 a 700 arbres et du sous-echantillonnage lignes/colonnes.


In [1]:
import sys
from pathlib import Path
ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from imblearn.combine import SMOTETomek
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.under_sampling import RandomUnderSampler
from sklearn.base import clone
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from xgboost import XGBClassifier


In [2]:
train = pd.read_csv(ROOT / 'data' / 'processed' / 'train.csv')
X_train, y_train = train.drop(columns='bad_nutrition'), train['bad_nutrition'].astype(int)
preprocessor = joblib.load(ROOT / 'models' / 'preprocessor.joblib')
selection = pd.read_csv(ROOT / 'models' / 'model_selection_results.csv')
best_selection = selection.iloc[0]
class_ratio = float((y_train == 0).sum() / (y_train == 1).sum())

if best_selection['model'] != 'XGBoost':
    raise ValueError('Le meilleur modele attendu pour le tuning est XGBoost.')

samplers = {
    'baseline': None,
    'smote': SMOTE(random_state=42),
    'undersample': RandomUnderSampler(random_state=42),
    'smote_tomek': SMOTETomek(random_state=42),
}
selected_strategy = str(best_selection['strategy'])
steps = [('preprocessor', clone(preprocessor))]
if samplers[selected_strategy] is not None:
    steps.append(('sampler', samplers[selected_strategy]))
steps.append(('clf', XGBClassifier(
        scale_pos_weight=class_ratio if selected_strategy == 'baseline' else 1.0,
        n_jobs=1,
        random_state=42,
        eval_metric='logloss',
    )))
pipeline = ImbPipeline(steps)
param_distributions = {
    'clf__n_estimators': [300, 500, 700],
    'clf__max_depth': [3, 5, 7],
    'clf__learning_rate': [0.03, 0.05, 0.1],
    'clf__subsample': [0.7, 0.85, 1.0],
    'clf__colsample_bytree': [0.7, 0.85, 1.0],
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
search = RandomizedSearchCV(
    pipeline,
    param_distributions=param_distributions,
    n_iter=10,
    scoring='f1',
    cv=cv,
    n_jobs=1,
    random_state=42,
    verbose=1,
    return_train_score=True,
)
search.fit(X_train, y_train)

tuned_artifact = {
    'pipeline': search.best_estimator_,
    'best_params': search.best_params_,
    'baseline_f1_mean': float(best_selection['mean_f1']),
    'tuned_f1_mean': float(search.best_score_),
    'imbalance_strategy': selected_strategy,
}
joblib.dump(tuned_artifact, ROOT / 'models' / 'tuned_model.joblib')

display(pd.Series({
    'baseline_f1_mean': tuned_artifact['baseline_f1_mean'],
    'tuned_f1_mean': tuned_artifact['tuned_f1_mean'],
}, name='F1 CV'))
display(pd.Series(search.best_params_, name='Valeur optimale'))
print('Modele tune serialise dans models/tuned_model.joblib')


Fitting 5 folds for each of 10 candidates, totalling 50 fits


baseline_f1_mean    0.886777
tuned_f1_mean       0.895360
Name: F1 CV, dtype: float64

clf__subsample             0.85
clf__n_estimators        500.00
clf__max_depth             7.00
clf__learning_rate         0.10
clf__colsample_bytree      1.00
Name: Valeur optimale, dtype: float64

Modele tune serialise dans models/tuned_model.joblib


## Synthese
Le tuning ameliore le F1 moyen en validation croisee. Le seuil de decision
n'est pas optimise ici: il est choisi ensuite sur le jeu de validation selon
la matrice de cout metier.
